# Día 3 — El sistema de Lorenz

### Taller: Física no lineal en el aula
**Congreso de Profesores de Física — Educación Secundaria**

---

Cerramos el taller juntando las dos mitades:

- el **Día 1** fueron ecuaciones diferenciales (péndulos), con sensibilidad a las
  condiciones iniciales,
- el **Día 2** fue un mapa iterado, con la ruta al caos y un número universal.

Hoy vamos a un sistema de ecuaciones diferenciales en 3D — y al final vamos a ver
que adentro tiene escondido **un mapa unidimensional muy parecido al logístico**.
Ése es el cierre del círculo.

---
**Recordatorio de Colab:** *Copiar en Drive* → *Entorno de ejecución → Ejecutar
todas* → esperar. Las celdas van en orden.

---
## 0. Preparación

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D          # noqa: activa el 3D
from ipywidgets import interact, FloatSlider, IntSlider

plt.rcParams["figure.figsize"] = (7, 4.2)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# parámetros clásicos de Lorenz (1963)
SIGMA, RHO, BETA = 10.0, 28.0, 8.0/3.0

print("Todo listo ✓")

---
## 1. De dónde salen las ecuaciones

En 1963 Edward Lorenz estudiaba un modelo **muy** simplificado de convección
atmosférica: una capa de aire calentada por abajo y enfriada por arriba. Quedándose
sólo con los tres modos más importantes del movimiento, llegó a

$$\dot{x} = \sigma(y - x)$$
$$\dot{y} = x(\rho - z) - y$$
$$\dot{z} = xy - \beta z$$

Las variables **no** son posiciones en el espacio:

| variable | significado físico |
|---|---|
| $x$ | intensidad del movimiento convectivo (los rollos de aire) |
| $y$ | diferencia de temperatura entre corrientes ascendentes y descendentes |
| $z$ | cuánto se aparta el perfil vertical de temperatura de una recta |

Los parámetros clásicos son $\sigma = 10$, $\rho = 28$, $\beta = 8/3$.

Fíjense en algo: **las únicas no linealidades son los productos $xz$ y $xy$.** Dos
multiplicaciones. Todo lo demás es lineal.

> **La historia vale la pena contarla.** Lorenz descubrió el fenómeno por
> accidente: quiso repetir una simulación y, para no empezar de cero, tipeó los
> números de una impresión intermedia. La impresión tenía 3 decimales; la
> computadora trabajaba con 6. Esa diferencia de una parte en mil produjo un
> pronóstico completamente distinto.
>
> Es **exactamente** el Ejercicio 2 del Día 1, cuando los dos mapas de cuencas
> calculados con distinto paso nos dieron dibujos diferentes. A Lorenz le pasó de
> verdad, en 1961, y fundó un campo entero.

In [ ]:
def lorenz(S, sigma=SIGMA, rho=RHO, beta=BETA):
    "S puede ser un punto (3,) o muchos puntos a la vez (3, N)."
    x, y, z = S
    return np.stack([sigma*(y - x),
                     x*(rho - z) - y,
                     x*y - beta*z])

def paso_rk4(S, dt, **kw):
    k1 = lorenz(S,           **kw)
    k2 = lorenz(S + dt/2*k1, **kw)
    k3 = lorenz(S + dt/2*k2, **kw)
    k4 = lorenz(S + dt*k3,   **kw)
    return S + dt/6*(k1 + 2*k2 + 2*k3 + k4)

def integrar(S, dt, tmax, **kw):
    n = int(tmax/dt)
    salida = np.empty((n+1,) + S.shape)
    salida[0] = S
    for i in range(n):
        S = paso_rk4(S, dt, **kw)
        salida[i+1] = S
    return np.linspace(0, n*dt, n+1), salida

### Un aviso sobre el paso de integración

Antes de dibujar nada, conviene verificar que el integrador esté bien. Comparamos
la posición a $t = 10$ calculada con distintos pasos $dt$.

In [ ]:
print("posición en t = 10 según el paso usado:\n")
ref = None
for dt in [0.02, 0.01, 0.005, 0.002, 0.001]:
    _, tr = integrar(np.array([1.0, 1.0, 1.0]), dt, 10.0)
    p = tr[-1]
    if ref is None:
        ref = p
        print(f"  dt = {dt:<6}  {np.round(p, 5)}")
    else:
        print(f"  dt = {dt:<6}  {np.round(p, 5)}   (difiere del primero en {np.linalg.norm(p-ref):.1e})")

De `dt = 0.005` para abajo el resultado ya no cambia: el integrador **convergió**.
Con `dt = 0.02` todavía hay error visible. Vamos a usar `0.005`.

> **Costumbre que vale la pena transmitir:** antes de sacar conclusiones de una
> simulación, hay que verificar que el resultado no dependa del paso. Es el
> equivalente numérico de calibrar un instrumento.

---
## 2. El atractor

Ahora sí: integramos y miramos la trayectoria en el espacio $(x, y, z)$.

In [ ]:
ts, tr = integrar(np.array([1.0, 1.0, 1.0]), 0.005, 100.0)

fig = plt.figure(figsize=(8, 6.5))
ax = fig.add_subplot(111, projection="3d")
ax.plot(tr[:,0], tr[:,1], tr[:,2], lw=0.4, color="steelblue")

# puntos fijos C± = (±√(β(ρ−1)), ±√(β(ρ−1)), ρ−1)
c = np.sqrt(BETA*(RHO - 1))
ax.scatter([c, -c], [c, -c], [RHO-1, RHO-1], color="crimson", s=45, zorder=5)
ax.scatter([0], [0], [0], color="black", s=35)

ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax.set_title("Atractor de Lorenz")
plt.show()

print(f"puntos fijos:  C± = (±{c:.4f}, ±{c:.4f}, {RHO-1:.0f})   y el origen")

Las dos alas no son "órbitas" alrededor de los puntos rojos: la trayectoria
**nunca se cierra** y **nunca se cruza a sí misma**. Da unas vueltas de un lado,
salta al otro, vuelve — sin patrón que se repita.

Los tres puntos fijos son **todos inestables**. No hay ningún equilibrio al que el
sistema pueda quedarse: eso es lo que lo mantiene en movimiento para siempre.

Comparen con el Día 1: en el péndulo con rozamiento el atractor era **un punto**;
en el péndulo magnético eran **tres puntos**. Acá el atractor es un objeto con
estructura, de dimensión fraccionaria (≈ 2.06). Por eso se lo llama
**atractor extraño**.

### 2.1 Las proyecciones

En 3D es difícil ver qué pasa. Miremos las sombras sobre los tres planos.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, (i, j, ni, nj) in zip(axes, [(0,1,"x","y"), (0,2,"x","z"), (1,2,"y","z")]):
    ax.plot(tr[:,i], tr[:,j], lw=0.3, color="steelblue")
    ax.set_xlabel(ni); ax.set_ylabel(nj); ax.set_title(f"plano {ni}–{nj}")
plt.tight_layout(); plt.show()

La proyección $x$–$z$ es la clásica "mariposa" (o "máscara"). Fíjense que en el
plano $x$–$y$ las dos alas se ven casi como una recta: es que $x$ e $y$ están muy
correlacionadas — la primera ecuación, $\dot x = \sigma(y-x)$, arrastra $x$ hacia $y$.

### 2.2 Explorar los parámetros

$\rho$ es el que manda: mide cuánto se calienta la capa de aire por abajo.

In [ ]:
@interact(rho=FloatSlider(min=0.5, max=60.0, step=0.5, value=28.0,
                          description="ρ", continuous_update=False),
          tmax=IntSlider(min=20, max=150, step=10, value=80,
                          description="t max", continuous_update=False))
def explorar_rho(rho, tmax):
    _, T = integrar(np.array([1.0, 1.0, 1.0]), 0.005, float(tmax), rho=rho)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2))
    a1.plot(T[:,0], T[:,2], lw=0.4, color="steelblue")
    a1.set_xlabel("x"); a1.set_ylabel("z"); a1.set_title(f"plano x–z    ρ = {rho}")
    a2.plot(np.linspace(0, tmax, len(T)), T[:,0], lw=0.6, color="darkslategray")
    a2.set_xlabel("t"); a2.set_ylabel("x(t)"); a2.set_title("serie temporal de x")
    plt.tight_layout(); plt.show()

✏️ **Para probar, en este orden:**

1. **$\rho = 0.5$** — todo muere en el origen. Sin convección: el aire no se mueve,
   el calor sube por conducción nomás.
2. **$\rho = 5$** — la trayectoria termina en uno de los dos puntos rojos.
   Convección **estacionaria**: rollos de aire girando siempre igual. El sentido de
   giro (izquierda o derecha) depende de la condición inicial.
3. **$\rho = 20$** — sigue terminando en un punto fijo, pero da muchas vueltas
   antes. Es un **transitorio largo**.
4. **$\rho = 24.5$** — ya casi no se decide.
5. **$\rho = 28$** — caos. La serie temporal de $x$ muestra los saltos entre alas,
   sin patrón.
6. **$\rho = 35$, $\rho = 42$** — sigue siendo caótico, pero cambia la forma.

> Miren la serie temporal de $x$ en $\rho = 28$: se parece bastante a un registro
> de datos experimentales con ruido. Sólo que **no hay ninguna aleatoriedad** en
> las ecuaciones. Es el mensaje del Día 1, otra vez.

---
## 3. Sensibilidad: el efecto mariposa

Igual que hicimos con el péndulo magnético, soltamos **dos trayectorias casi
idénticas** y miramos cuánto tardan en separarse.

In [ ]:
@interact(delta=FloatSlider(min=-12, max=-2, step=1, value=-9,
                            description="log₁₀(δ)", continuous_update=False),
          tmax=IntSlider(min=20, max=80, step=5, value=45,
                            description="t max", continuous_update=False))
def dos_trayectorias(delta, tmax):
    d0 = 10.0**delta
    S = np.array([[1.0, 1.0 + d0], [1.0, 1.0], [1.0, 1.0]])   # (3, 2)
    t, T = integrar(S, 0.005, float(tmax))

    sep = np.linalg.norm(T[:,:,0] - T[:,:,1], axis=1)

    fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 7))
    a1.plot(t, T[:,0,0], lw=0.8, color="black",  label="trayectoria A")
    a1.plot(t, T[:,0,1], lw=0.8, color="orange", label="trayectoria B")
    a1.set_ylabel("x(t)"); a1.legend(loc="upper right", fontsize=8)
    a1.set_title(f"separación inicial δ = {d0:.0e}")

    a2.semilogy(t, np.maximum(sep, 1e-16), lw=1.2, color="crimson")
    a2.axhline(30, ls="--", color="gray")
    a2.set_xlabel("t"); a2.set_ylabel("distancia entre A y B")
    a2.text(t[-1]*0.02, 40, "tamaño del atractor", fontsize=8, color="gray")
    plt.tight_layout(); plt.show()

    sup = np.where(sep > 30)[0]
    if len(sup):
        print(f"→ Las trayectorias se vuelven independientes a t ≈ {t[sup[0]]:.1f}")

✏️ **Para probar:** muevan $\delta$ de $10^{-2}$ a $10^{-12}$ y miren el **panel de
abajo**, que está en escala logarítmica.

Van a ver dos cosas:

1. La separación crece en **línea recta** en escala logarítmica. Recta en semilog
   significa **crecimiento exponencial**: $d(t) \approx d_0\,e^{\lambda t}$.
2. Achicar $\delta$ mil veces **no** da mil veces más tiempo de predicción: sólo
   corre la recta un poco hacia la derecha. Cada factor 10 de precisión extra
   compra siempre el **mismo** ratito adicional.

> **Ésta es la razón profunda de que el pronóstico del tiempo tenga un horizonte.**
> No es que falten estaciones meteorológicas. Es que mejorar los datos da
> rendimientos logarítmicos: para duplicar el horizonte de predicción habría que
> elevar la precisión al cuadrado.

---
## 4. El exponente de Lyapunov

Esa recta del gráfico anterior tiene una pendiente, y esa pendiente es un número
que caracteriza al sistema: el **exponente de Lyapunov** $\lambda$.

$$d(t) \approx d_0\,e^{\lambda t}
\qquad\Longrightarrow\qquad
\lambda = \lim_{t\to\infty}\frac{1}{t}\ln\frac{d(t)}{d_0}$$

- $\lambda > 0$ → **caos** (las diferencias se amplifican),
- $\lambda \approx 0$ → **órbita periódica** (ciclo límite),
- $\lambda < 0$ → **punto fijo**.

> ⚠️ **Ojo con una diferencia respecto del Día 2.** En un mapa iterado, una órbita
> periódica estable da $\lambda < 0$. En un sistema **continuo** como éste, da
> $\lambda \approx 0$: siempre existe la dirección *a lo largo* de la trayectoria,
> en la que dos puntos vecinos ni se acercan ni se alejan. Sólo los puntos fijos
> dan $\lambda$ netamente negativo.

### El problema, y cómo se resuelve

No podemos simplemente medir $d(t)$ a tiempo largo: las trayectorias se separan
hasta el tamaño del atractor y ahí la distancia **deja de crecer**. La recta se
aplana y arruina la medición.

La solución es el **algoritmo de Benettin**: cada paso se mide cuánto creció la
separación, se anota el logaritmo, y **se vuelve a acercar** la segunda trayectoria
a la primera manteniendo la dirección. Así nunca se sale del régimen exponencial.

Es un truco muy visual: se mide el estiramiento de a pedacitos y se van sumando.

In [ ]:
def lyapunov(dt=0.005, tmax=1000.0, d0=1e-8, t_transitorio=100.0,
             sigma=SIGMA, rho=RHO, beta=BETA):
    "Exponente de Lyapunov máximo por el algoritmo de Benettin."

    # versión escalar del paso: es ~60 veces más rápida que usar
    # arrays de numpy de 3 elementos, porque evita todo el overhead
    def f(x, y, z):
        return sigma*(y - x), x*(rho - z) - y, x*y - beta*z

    def paso(x, y, z):
        a1, b1, c1 = f(x, y, z)
        a2, b2, c2 = f(x + dt/2*a1, y + dt/2*b1, z + dt/2*c1)
        a3, b3, c3 = f(x + dt/2*a2, y + dt/2*b2, z + dt/2*c2)
        a4, b4, c4 = f(x + dt*a3,   y + dt*b3,   z + dt*c3)
        return (x + dt/6*(a1 + 2*a2 + 2*a3 + a4),
                y + dt/6*(b1 + 2*b2 + 2*b3 + b4),
                z + dt/6*(c1 + 2*c2 + 2*c3 + c4))

    # 1) descartamos el transitorio: queremos medir SOBRE el atractor
    x, y, z = 1.0, 1.0, 1.0
    for _ in range(int(t_transitorio/dt)):
        x, y, z = paso(x, y, z)

    # 2) soltamos una segunda trayectoria a distancia d0
    X, Y, Z = x + d0, y, z

    suma = 0.0
    n = int(tmax/dt)
    for _ in range(n):
        x, y, z = paso(x, y, z)
        X, Y, Z = paso(X, Y, Z)

        dx, dy, dz = X - x, Y - y, Z - z
        d = (dx*dx + dy*dy + dz*dz)**0.5

        suma += np.log(d/d0)          # anotamos cuánto se estiró

        k = d0/d                      # y la volvemos a acercar,
        X, Y, Z = x + dx*k, y + dy*k, z + dz*k   # sin cambiar la dirección

    return suma/(n*dt)

In [ ]:
print("convergencia con el tiempo de medición:\n")
for tmax in [100, 300, 500, 1000, 2000]:
    print(f"  t = {tmax:5d}   λ = {lyapunov(tmax=float(tmax)):.4f}")

print("\n  valor de referencia en la literatura:  λ ≈ 0.9056")

El valor converge a **λ ≈ 0.90**, que coincide con el de la literatura.

Dos controles para convencerse de que el número mide lo que dice medir:

In [ ]:
print("control 1 — un régimen NO caótico (ρ = 13, termina en un punto fijo):")
print(f"    λ = {lyapunov(tmax=500.0, rho=13.0):+.4f}   → negativo, como debe ser\n")

print("control 2 — el mismo caso caótico con otro paso de integración:")
print(f"    dt = 0.005 → λ = {lyapunov(tmax=500.0):+.4f}")
print(f"    dt = 0.002 → λ = {lyapunov(dt=0.002, tmax=500.0):+.4f}")
print("    → el resultado no depende del paso: estamos midiendo física, no ruido numérico")

### ¿Qué significa 0.90 en la práctica?

El **tiempo de duplicación** de un error es $t_2 = \ln 2/\lambda$.

In [ ]:
lam = 0.9056
t2  = np.log(2)/lam
print(f"tiempo de duplicación del error:  ln2/λ = {t2:.3f} unidades de tiempo\n")
for factor in [10, 100, 1000]:
    print(f"  para que un error se multiplique por {factor:4d}: "
          f"{np.log(factor)/lam:.2f} unidades de tiempo")
print(f"\n  y para ganar UN factor 10 más de horizonte, siempre hace falta")
print(f"  mejorar la precisión inicial por 10. Rendimiento logarítmico.")

🧩 **Ejercicio 1.** ¿Para qué valor de $\rho$ el sistema **deja** de ser caótico?

Cambien el valor de abajo y busquen dónde $\lambda$ deja de ser positivo.

*(Pista: hay una transición bastante brusca **entre 23 y 24**. Y después de la
zona caótica, más arriba de lo que uno esperaría — prueben $\rho = 100$, $150$,
$160$ — el sistema vuelve a ser periódico. Ojo: $\rho = 40$ o $45$ **siguen**
siendo caóticos.)*

In [ ]:
rho_prueba = 28.0      # ←←← CAMBIAR ESTE NÚMERO

lam = lyapunov(tmax=600.0, rho=rho_prueba)
if   lam >  0.02: estado = "CAÓTICO"
elif lam < -0.02: estado = "punto fijo (todo se frena)"
else:             estado = "órbita periódica (ciclo límite)"
print(f"ρ = {rho_prueba}   →   λ = {lam:+.4f}   →   {estado}")

---
## 5. El cierre del círculo: Lorenz esconde un mapa

Y ahora la parte que conecta los tres días.

Lorenz se hizo esta pregunta: *si anoto el valor **máximo** que alcanza $z$ en cada
vuelta, ¿el próximo máximo depende del anterior?*

Es decir: llamemos $z_1, z_2, z_3, \ldots$ a los máximos sucesivos de $z(t)$, y
grafiquemos $z_{n+1}$ contra $z_n$.

Si el sistema fuera realmente "aleatorio", eso tendría que dar una **nube de
puntos sin estructura**. Antes de ejecutar, hagan su apuesta.

In [ ]:
def maximos_de_z(dt=0.002, tmax=600.0, t_transitorio=50.0, **kw):
    "Devuelve los máximos locales sucesivos de z(t), sobre el atractor."
    S = np.array([1.0, 1.0, 1.0])
    for _ in range(int(t_transitorio/dt)):      # descartar transitorio
        S = paso_rk4(S, dt, **kw)

    n = int(tmax/dt)
    z = np.empty(n+1); z[0] = S[2]
    for i in range(n):
        S = paso_rk4(S, dt, **kw)
        z[i+1] = S[2]

    # máximos locales, afinados con una parábola por los 3 puntos vecinos
    idx = np.where((z[1:-1] > z[:-2]) & (z[1:-1] > z[2:]))[0] + 1
    picos = []
    for i in idx:
        a, b, c = z[i-1], z[i], z[i+1]
        den = a - 2*b + c
        picos.append(b - 0.125*(c - a)**2/den if den != 0 else b)
    return np.array(picos)

zm = maximos_de_z()
print(f"{len(zm)} máximos encontrados, entre {zm.min():.2f} y {zm.max():.2f}")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5.2))

a1.plot(zm[:-1], zm[1:], ".", ms=3, color="crimson")
lim = [zm.min()-0.5, zm.max()+0.5]
a1.plot(lim, lim, ls="--", lw=1, color="gray")
a1.set_xlabel("zₙ  (máximo actual)"); a1.set_ylabel("zₙ₊₁  (máximo siguiente)")
a1.set_title("El mapa de Lorenz")
a1.set_xlim(lim); a1.set_ylim(lim); a1.set_aspect("equal")

x = np.linspace(0, 1, 300)
a2.plot(x, 3.9*x*(1-x), lw=2, color="steelblue")
a2.plot([0,1], [0,1], ls="--", lw=1, color="gray")
a2.set_xlabel("xₙ"); a2.set_ylabel("xₙ₊₁")
a2.set_title("El mapa logístico de ayer (r = 3.9)")
a2.set_aspect("equal")

plt.tight_layout(); plt.show()

**No es una nube: es una curva.** Con un pico, igual que la parábola de ayer.

Esto es notable. Tenemos un sistema de **tres ecuaciones diferenciales acopladas**,
con trayectorias que nunca se repiten, y sin embargo una sola pregunta bien elegida
("¿cuál es el próximo máximo de $z$?") lo reduce a **una función de una variable**.

Ésa es la razón de haber pasado un día entero con $x(1-x)$: el mapa logístico no es
un juguete aparte, es la **estructura mínima** que aparece adentro de sistemas
mucho más complicados.

Midamos qué tan buena es esa reducción:

In [ ]:
# ¿qué tan "fina" es la curva? Comparamos su grosor con su extensión.
bins = np.linspace(zm.min(), zm.max(), 40)
grosores = []
for i in range(len(bins)-1):
    m = (zm[:-1] >= bins[i]) & (zm[:-1] < bins[i+1])
    if m.sum() > 5:
        grosores.append(zm[1:][m].std())

grosor = np.median(grosores)
rango  = zm.max() - zm.min()
print(f"grosor típico de la curva : {grosor:.3f}")
print(f"extensión total           : {rango:.3f}")
print(f"→ la curva es {100*grosor/rango:.1f} % de ancha: es una curva, no una nube\n")

# la pendiente: si |f'| > 1 en todos lados, todo error se amplifica
orden = np.argsort(zm[:-1])
a, b = zm[:-1][orden], zm[1:][orden]
pend = np.abs(np.diff(b)/np.diff(a))
pend = pend[np.isfinite(pend)]
print(f"pendiente |f'| típica: {np.median(pend):.2f}")
print("→ mayor que 1 en casi todo el dominio: por eso el sistema es caótico.")

El último número es la explicación completa del caos de Lorenz, en el lenguaje que
construimos ayer: **la pendiente del mapa es mayor que 1 en casi todo el dominio**.

En el Día 2 vimos que un punto fijo es estable si $|f'| < 1$ e inestable si
$|f'| > 1$. Acá $|f'| > 1$ en **todos** lados: no hay ningún punto fijo estable
posible, ninguna órbita periódica que sobreviva, y cualquier diferencia entre dos
condiciones iniciales se amplifica en cada vuelta.

El atractor extraño en 3D y la parábola de la calculadora son **el mismo fenómeno**.

🧩 **Ejercicio 2 — reconstruir un sistema a partir de una sola señal**

Éste es un buen ejercicio final porque invierte el punto de vista.

Imaginen que sólo tienen un sensor, que mide **una** variable: $z(t)$. No conocen
las ecuaciones. ¿Se puede decir algo?

La respuesta es que sí: graficando $z(t)$ contra $z(t - \tau)$ para algún retardo
$\tau$, se **reconstruye** una figura equivalente al atractor. Es el teorema de
Takens, y es la base de casi todo el análisis de series temporales caóticas
experimentales.

Prueben distintos retardos.

In [ ]:
retardo = 0.15      # ←←← CAMBIAR (probar 0.02, 0.08, 0.15, 0.4, 1.0)

dt = 0.005
_, T = integrar(np.array([1.0, 1.0, 1.0]), dt, 120.0)
z = T[int(20/dt):, 2]                  # descartamos el transitorio
k = int(retardo/dt)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5))
a1.plot(np.arange(len(z))*dt, z, lw=0.5, color="darkslategray")
a1.set_xlabel("t"); a1.set_ylabel("z(t)")
a1.set_title("la única señal que tenemos")

a2.plot(z[:-k], z[k:], lw=0.35, color="purple")
a2.set_xlabel("z(t)"); a2.set_ylabel(f"z(t + {retardo})")
a2.set_title(f"atractor reconstruido    τ = {retardo}")
plt.tight_layout(); plt.show()

✏️ **Para probar:** con $\tau$ muy chico (0.02) todo se aplasta contra la diagonal —
$z(t)$ y $z(t+\tau)$ son casi el mismo número. Con $\tau$ muy grande (1.0) se
enreda y pierde forma. En el medio (≈ 0.15) aparece una figura con estructura clara.

Lo importante: esa figura **no es** el atractor de Lorenz, pero es equivalente a él
en las propiedades que importan (dimensión, exponentes de Lyapunov). Y se obtuvo
**sin conocer las ecuaciones**, con un solo sensor.

> **Para el aula:** esto es lo que hace utilizable toda la teoría del taller con
> datos experimentales reales — un termistor, un micrófono, un sensor de presión.

---
## 6. Cierre de los tres días

| | Día 1 | Día 2 | Día 3 |
|---|---|---|---|
| **Sistema** | péndulos | mapa logístico | Lorenz |
| **Tipo** | EDO, 2D y 4D | mapa iterado, 1D | EDO, 3D |
| **Atractor** | punto(s) | ciclos y caos | atractor extraño |
| **Herramienta** | espacio de fases, cuencas | diagrama de bifurcación | Lyapunov, mapa de retorno |
| **Número medido** | — | $\delta = 4.669$ | $\lambda = 0.906$ |

**Las tres ideas del taller:**

1. **La complejidad no requiere ecuaciones complicadas.** El mapa logístico es
   $x(1-x)$; Lorenz tiene dos productos. Lo que genera la riqueza no es la
   complicación algebraica sino la no linealidad más una realimentación.

2. **Determinista no es predecible.** Sin ninguna aleatoriedad en las ecuaciones,
   la predicción tiene un horizonte finito — y mejorar los datos da rendimientos
   logarítmicos.

3. **El caos es medible.** No es una etiqueta cualitativa: $\lambda$ y $\delta$ son
   números, se calculan, se comparan con experimentos y algunos son universales.

**Y una idea para el aula:** casi todo esto se puede hacer con una calculadora, una
planilla de cálculo o un cuaderno como éste. El mapa logístico entra en una clase.
El péndulo magnético es un objeto de escritorio. La barrera de entrada es mucho más
baja de lo que el tema sugiere.

---
### Para seguir

- **Strogatz**, *Nonlinear Dynamics and Chaos* — el libro de referencia, muy legible.
- **Gleick**, *Chaos: Making a New Science* — el relato histórico, sin matemática.
- **Lorenz (1963)**, *Deterministic Nonperiodic Flow* — el artículo original, y es
  sorprendentemente claro.
- **Feigenbaum (1978)**, *Quantitative universality for a class of nonlinear
  transformations*.
- **Li & Yorke (1975)**, *Period three implies chaos* — el que le puso nombre al campo.